# Sber sbert_large_mt_nlu_ru — 30 rare words

GPU accelerator ON. Same vocab 38 360 words, same method as previous test. Output: `sbert_rare30.json`.

In [ ]:
!pip install -q sentence-transformers
import json, time, numpy as np, urllib.request
from sentence_transformers import SentenceTransformer

In [ ]:
url = 'https://raw.githubusercontent.com/alex-mtl/mao-dao/main/stream-engine/word-data/built/ru/vocab.txt'
vocab = [w.strip() for w in urllib.request.urlopen(url).read().decode('utf-8').splitlines() if w.strip()]
print('vocab', len(vocab))

In [ ]:
SECRETS = {
 "абажур": [
  "лампа",
  "светильник"
 ],
 "авантюрист": [
  "аферист",
  "проходимец"
 ],
 "балагур": [
  "шутник",
  "весельчак"
 ],
 "барышник": [
  "торговец",
  "спекулянт"
 ],
 "бедолага": [
  "бедняга",
  "горемыка"
 ],
 "бобыль": [
  "холостяк",
  "одиночка"
 ],
 "болван": [
  "дурак",
  "глупец"
 ],
 "бражник": [
  "пьяница",
  "кутила"
 ],
 "бузотер": [
  "хулиган",
  "дебошир"
 ],
 "варяг": [
  "наемник",
  "воин"
 ],
 "вещун": [
  "предсказатель",
  "пророк"
 ],
 "волокита": [
  "бюрократия",
  "проволочка"
 ],
 "ворожея": [
  "гадалка",
  "колдунья"
 ],
 "выскочка": [
  "зазнайка",
  "карьерист"
 ],
 "гаер": [
  "шут",
  "паяц"
 ],
 "галоп": [
  "скачка",
  "аллюр"
 ],
 "гармонь": [
  "аккордеон",
  "гармоника"
 ],
 "глухарь": [
  "тетерев",
  "птица"
 ],
 "головорез": [
  "бандит",
  "разбойник"
 ],
 "горлопан": [
  "крикун",
  "скандалист"
 ],
 "гравер": [
  "резчик",
  "художник"
 ],
 "гребец": [
  "лодочник",
  "спортсмен"
 ],
 "грибник": [
  "собиратель",
  "любитель"
 ],
 "гувернер": [
  "воспитатель",
  "наставник"
 ],
 "гурман": [
  "ценитель",
  "лакомка"
 ],
 "дачник": [
  "садовод",
  "огородник"
 ],
 "касик": [
  "вождь",
  "правитель"
 ],
 "котом": [
  "кот",
  "котик"
 ],
 "шеллак": [
  "лак",
  "смола"
 ],
 "неувязка": [
  "несоответствие",
  "нестыковка"
 ]
}

In [ ]:
model = SentenceTransformer('ai-forever/sbert_large_mt_nlu_ru')
print('dim', model.get_embedding_dimension())
t0 = time.time()
emb = model.encode(vocab, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
print(f'vocab encoded in {time.time()-t0:.0f}s shape {emb.shape}')
emb = np.asarray(emb, dtype=np.float32)
idx = {w: i for i, w in enumerate(vocab)}
out = {}
for secret, syns in SECRETS.items():
    sims = emb @ emb[idx[secret]]
    order = np.argsort(-sims, kind='stable')
    top = [(vocab[int(i)], int(r+1)) for r, i in enumerate(order[:30])]
    syn = {s: int(np.where(order == idx[s])[0][0]) + 1 for s in syns if s in idx}
    out[secret] = {'after_top30': top, 'after_syn_ranks': syn}
    print(f"AFTER {secret}: top={[w for w, _ in top[:6]]} syn={syn}", flush=True)
json.dump({'secrets': out}, open('sbert_rare30.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('wrote sbert_rare30.json')